In [5]:
!pip install feedparser

In [6]:
import feedparser
import html
from transformers import pipeline
import requests

In [7]:
MAX_ARTICLES_PER_FEED = 2

RSS_FEEDS = [
    "https://techcrunch.com/feed/",
    "https://www.theverge.com/rss/index.xml",
    "https://www.wired.com/feed/category/gear/latest/rss"
]

In [8]:
def clean_text(text):
    return html.unescape(text).encode("utf-8", errors="ignore").decode("utf-8")

In [9]:
def get_news():
    news_items = []
    for feed_url in RSS_FEEDS:
        feed = feedparser.parse(feed_url)
        articles = feed.entries[:MAX_ARTICLES_PER_FEED]
        for a in articles:
            title = clean_text(a.get("title", "No title"))
            link = clean_text(a.get("link", "No link"))
            summary = clean_text(a.get("summary", "No summary"))
            news_items.append({"title": title, "link": link, "summary": summary})
    return news_items

In [13]:
def summarize_with_huggingface(news_items):
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    text = " ".join([n["title"] + ". " + n["summary"] for n in news_items])
    result = summarizer(text, max_length=150, min_length=60, do_sample=False)
    summary = result[0]['summary_text']
    post = "🚀 Tech Updates Today:\n\n"
    post += summary + "\n\n"
    post += "🔗 Links:\n"
    for n in news_items:
        post += f"- {n['title']}: {n['link']}\n"
    return post

In [14]:
def save_post(content):
    print("\n✅ --- متن نهایی پست --- ✅\n")
    print(content)
    with open("post_ready.txt", "w", encoding="utf-8") as f:
        f.write(content)
    print("\n📂 متن آماده در فایل post_ready.txt ذخیره شد.\n")

In [15]:
if __name__ == "__main__":
    news = get_news()
    post = summarize_with_huggingface(news)
    save_post(post)

Device set to use cpu



✅ --- متن نهایی پست --- ✅

🚀 Tech Updates Today:

Logitech’s new light-powered keyboard doesn’t even need the sun. Jimmy Kimmel returns to television to mock FCC Chair Brendan Carr. OpenAI is building five new Stargate data centers with Oracle and SoftBank. Building the new backbone of space at TechCrunch Disrupt 2025.

🔗 Links:
- OpenAI is building five new Stargate data centers with Oracle and SoftBank: https://techcrunch.com/2025/09/23/openai-is-building-five-new-stargate-data-centers-with-oracle-and-softbank/
- Building the new backbone of space at TechCrunch Disrupt 2025: https://techcrunch.com/2025/09/23/space-is-open-for-business-with-even-rogers-and-max-haot-at-techcrunch-disrupt-2025/
- Logitech’s new light-powered keyboard doesn’t even need the sun: https://www.theverge.com/news/782968/logitech-signature-solar-k980-keyboard
- Kimmel returns to television to mock FCC Chair Brendan Carr: https://www.theverge.com/policy/784355/jimmy-kimmel-return-brendan-carr-trump
- Logitech S

In [16]:
TOKEN = "your token"
CHAT_ID = your chanal id
with open("post_ready.txt", "r", encoding="utf-8") as f:
    content = f.read()

url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
payload = {
    "chat_id": CHAT_ID,
    "text": content
}

res = requests.post(url, data=payload)
if res.status_code == 200:
    print("✅ پست با موفقیت ارسال شد!")
else:
    print("❌ خطا:", res.text)

✅ پست با موفقیت ارسال شد!
